In [ ]:
%load_ext autoreload
%autoreload 2
# import yfinance as yf
import pandas as pd
# import numpy as np
from datetime import date

from backtestConfig import BacktestConfig
from dataModule import DataModule, DataSource, CachedData
from executionModule import ExecutionModule
from strategyModule import StrategyModule, SMA
from resultsModule import ResultsModule
from portfolio_analytics import sharpe, drawdown_stats

from backtestEngine import BacktestEngine
from run_backtest import main

In [ ]:
cfg = BacktestConfig(start_date=date(2015, 1, 1), end_date=date(2026, 1, 1))
execution = ExecutionModule(cfg.fee_bps, cfg.slippage_bps)
results = ResultsModule(["sharpe", "drawdown", "sortino","vol", "return", "calmar"])

# simple parameter grid:
rows = []
for t in ["AAPL", "MSFT", "AMZN", "GOOGL", "NVDA", "GS"]:
    # get the data then cache
    data = DataModule(DataSource("yfinance", "historical", [t]))
    cache = CachedData(data.load(cfg.start_date, cfg.end_date, cfg.rebalance_freq))

    for w in [10, 20, 50, 100, 200]:
        for th in [0.005, 0.01, 0.03, 0.05, 0.1]:
            strategies = StrategyModule([SMA(window=w, threshold=th)])
    
            print("#" + 20 * "-" + f"{t}—Window: {w} | Threshold: {th}" + 20 * "-" + "#")
            engine = BacktestEngine(cfg, cache, strategies, execution, results)
            eq = engine.run()
            engine.display_results(eq, w)
            print("#" + 71 * "-" + "#")
            mdd, longest, _ = drawdown_stats(eq)
            rows.append({"ticker": t, "window": w, "thr": th, "sharpe": sharpe(eq.pct_change().dropna().to_numpy()), "maxdd": mdd,
            "trades": len(engine.trades), "ruined": bool(eq.min() <= 0)})
            
# eq -> is a pd.series with the correct index
group = pd.DataFrame(rows)

# print at the end
for ticker, group in group.groupby("ticker"):
    print(f"\n{ticker}")
    print(
        group.pivot_table(
            index="window",
            columns="thr",
            values="sharpe"
        )
    )

# aggregated
pd.DataFrame(rows).pivot_table(index="window", columns="thr", values="sharpe")

#--------------------AAPL—Window: 10 | Threshold: 0.005--------------------#
Sharpe (ann.): -0.54
Max drawdown: -94.4% | longest underwater: 2478 bars
Sortino: -0.77
Vol (ann.): 29.80%
Return (ann.): -18.58%
Calmar (ann.): -0.20
#-----------------------------------------------------------------------#
#--------------------AAPL—Window: 10 | Threshold: 0.01--------------------#
Sharpe (ann.): -0.58
Max drawdown: -94.2% | longest underwater: 2475 bars
Sortino: -0.83
Vol (ann.): 28.47%
Return (ann.): -18.64%
Calmar (ann.): -0.20
#-----------------------------------------------------------------------#
#--------------------AAPL—Window: 10 | Threshold: 0.03--------------------#
Sharpe (ann.): -0.31
Max drawdown: -84.0% | longest underwater: 2475 bars
Sortino: -0.45
Vol (ann.): 24.72%
Return (ann.): -10.20%
Calmar (ann.): -0.12
#-----------------------------------------------------------------------#
#--------------------AAPL—Window: 10 | Threshold: 0.05--------------------#
Sharpe (ann.): -0

/Users/main/Desktop/python-quant/backtest/backtestEngine.py:52: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if prev != signals[t]:
/Users/main/Desktop/python-quant/backtest/backtestEngine.py:53: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  units = signals[t] * c.max_leverage * max(pf.equity(px[t]), 0.0) / px[t]
/Users/main/Desktop/python-quant/backtest/backtestEngine.py:54: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  self._trad

Sharpe (ann.): -0.23
Max drawdown: -58169.6% | longest underwater: 1146 bars
Sortino: -0.26
Vol (ann.): 1487.45%
Return (ann.): 1136.32%
Calmar (ann.): 0.02
#-----------------------------------------------------------------------#
#--------------------AAPL—Window: 50 | Threshold: 0.03--------------------#
Sharpe (ann.): -0.08
Max drawdown: -24860.8% | longest underwater: 733 bars
Sortino: -0.09
Vol (ann.): 1322.95%
Return (ann.): 410.11%
Calmar (ann.): 0.02
#-----------------------------------------------------------------------#
#--------------------AAPL—Window: 50 | Threshold: 0.05--------------------#
Sharpe (ann.): -0.30
Max drawdown: -9342.2% | longest underwater: 915 bars
Sortino: -0.32
Vol (ann.): 1218.85%
Return (ann.): 262.78%
Calmar (ann.): 0.03
#-----------------------------------------------------------------------#
#--------------------AAPL—Window: 50 | Threshold: 0.1--------------------#
Sharpe (ann.): -0.24
Max drawdown: -842.7% | longest underwater: 902 bars
Sortino: -0

/Users/main/Desktop/python-quant/backtest/backtestEngine.py:52: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if prev != signals[t]:
/Users/main/Desktop/python-quant/backtest/backtestEngine.py:53: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  units = signals[t] * c.max_leverage * max(pf.equity(px[t]), 0.0) / px[t]
/Users/main/Desktop/python-quant/backtest/backtestEngine.py:54: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  self._trad

Sharpe (ann.): 0.33
Max drawdown: -340170.4% | longest underwater: 402 bars
Sortino: 16.68
Vol (ann.): 625388.78%
Return (ann.): 120987719813.34%
Calmar (ann.): 355667.99
#-----------------------------------------------------------------------#
#--------------------AAPL—Window: 200 | Threshold: 0.1--------------------#
Sharpe (ann.): 0.32
Max drawdown: -340170.4% | longest underwater: 396 bars
Sortino: 16.42
Vol (ann.): 625304.12%
Return (ann.): 10605216831.28%
Calmar (ann.): 31176.19
#-----------------------------------------------------------------------#
#--------------------MSFT—Window: 10 | Threshold: 0.005--------------------#
Sharpe (ann.): 0.21
Max drawdown: -49.3% | longest underwater: 1554 bars
Sortino: 0.30
Vol (ann.): 27.24%
Return (ann.): 1.91%
Calmar (ann.): 0.04
#-----------------------------------------------------------------------#
#--------------------MSFT—Window: 10 | Threshold: 0.01--------------------#
Sharpe (ann.): 0.32
Max drawdown: -45.9% | longest underwater:

/Users/main/Desktop/python-quant/backtest/backtestEngine.py:52: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if prev != signals[t]:
/Users/main/Desktop/python-quant/backtest/backtestEngine.py:53: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  units = signals[t] * c.max_leverage * max(pf.equity(px[t]), 0.0) / px[t]
/Users/main/Desktop/python-quant/backtest/backtestEngine.py:54: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  self._trad

... (21 outputs left out)

/Users/main/Desktop/python-quant/backtest/backtestEngine.py:52: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if prev != signals[t]:
/Users/main/Desktop/python-quant/backtest/backtestEngine.py:53: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  units = signals[t] * c.max_leverage * max(pf.equity(px[t]), 0.0) / px[t]
/Users/main/Desktop/python-quant/backtest/backtestEngine.py:54: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  self._trad

Sharpe (ann.): -0.28
Max drawdown: -71880.4% | longest underwater: 830 bars
Sortino: -0.28
Vol (ann.): 5661.59%
Return (ann.): 675.33%
Calmar (ann.): 0.01
#-----------------------------------------------------------------------#
#--------------------GS—Window: 100 | Threshold: 0.005--------------------#
Sharpe (ann.): -0.12
Max drawdown: -88239.5% | longest underwater: 665 bars
Sortino: -0.15
Vol (ann.): 51541.15%
Return (ann.): 6463562826.59%
Calmar (ann.): 73250.19
#-----------------------------------------------------------------------#
#--------------------GS—Window: 100 | Threshold: 0.01--------------------#
Sharpe (ann.): -0.13
Max drawdown: -79089.7% | longest underwater: 698 bars
Sortino: -0.17
Vol (ann.): 51489.59%
Return (ann.): 795732104.37%
Calmar (ann.): 10061.14
#-----------------------------------------------------------------------#
#--------------------GS—Window: 100 | Threshold: 0.03--------------------#
Sharpe (ann.): -0.13
Max drawdown: -79089.7% | longest underwate

/Users/main/Desktop/python-quant/backtest/backtestEngine.py:52: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if prev != signals[t]:
/Users/main/Desktop/python-quant/backtest/backtestEngine.py:53: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  units = signals[t] * c.max_leverage * max(pf.equity(px[t]), 0.0) / px[t]
/Users/main/Desktop/python-quant/backtest/backtestEngine.py:54: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  self._trad

thr,0.005,0.010,0.030,0.050,0.100
window,,,,,
10,-0.158417,-0.121516,-0.025127,0.105393,0.158890
20,-0.242539,-0.215487,-0.136916,-0.055865,0.074547
50,0.037172,-0.022796,0.004009,-0.004044,-0.019442
100,-0.015442,-0.016087,-0.011663,-0.029870,-0.007568
200,-0.038829,-0.037245,0.037430,0.105445,0.068649
